In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PROJECT_DIR = Path(r"D:\UPI_Fraud_Detection_Project_ML")
DATA_PATH = PROJECT_DIR / "data" / "upi_fraud_detection_cleaned.csv"

In [3]:
df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])

In [4]:
print("Dataset Shape:", df.shape)
display(df.head())

Dataset Shape: (100000, 22)


,transaction_id,user_id,timestamp,amount,merchant_category,transaction_type,sender_bank,receiver_bank,device_type,os,...,device_trust_score,failed_attempts,transaction_velocity,account_age_days,is_new_device,is_new_location,beneficiary_age_days,hour,is_weekend,is_fraud
0,TXN00000001,USR001339,2026-07-20 17:27:54,154.25,Food,Bill Payment,Kotak,SBI,Android,Android 15,...,91.88,0,4,2472,0,0,932,17,0,0
1,TXN00000002,USR011610,2026-08-29 20:08:54,102.42,Utilities,Bill Payment,Axis,Kotak,Android,Android 14,...,96.16,0,2,163,0,0,1166,20,1,0
2,TXN00000003,USR009819,2026-06-06 02:13:15,324.43,Travel,Merchant Payment,IDFC First,Kotak,Android,Android 15,...,77.42,1,2,368,0,0,392,2,1,0
3,TXN00000004,USR006584,2026-04-24 00:45:32,42.56,Shopping,Merchant Payment,IndusInd,IDFC First,Android,Android 14,...,96.22,1,3,953,0,1,490,0,0,0
4,TXN00000005,USR006496,2026-01-02 06:16:25,42.32,Education,Bank Transfer,IndusInd,ICICI,Android,Android 15,...,69.86,0,2,104,0,0,665,6,0,0


### Timestamp and features 

In [ ]:
df_fe = df.copy()

In [6]:
df_fe["day_of_week"] = df_fe["timestamp"].dt.dayofweek
df_fe["day_of_month"] = df_fe["timestamp"].dt.day
df_fe["month"] = df_fe["timestamp"].dt.month

In [ ]:
# 0 = Monday, 6 = Sunday
print("New features created:")
display(
    df_fe[
        [
            "timestamp",
            "hour",
            "day_of_week",
            "day_of_month",
            "month",
            "is_weekend",
            "is_fraud",
        ]
    ].head(10)
)

New features created:


,timestamp,hour,day_of_week,day_of_month,month,is_weekend,is_fraud
0,2026-07-20 17:27:54,17,0,20,7,0,0
1,2026-08-29 20:08:54,20,5,29,8,1,0
2,2026-06-06 02:13:15,2,5,6,6,1,0
3,2026-04-24 00:45:32,0,4,24,4,0,0
4,2026-01-02 06:16:25,6,4,2,1,0,0
5,2026-06-17 06:11:35,6,2,17,6,0,0
6,2026-09-05 21:55:45,21,5,5,9,1,0
7,2026-02-08 10:31:45,10,6,8,2,1,0
8,2026-03-08 16:44:55,16,6,8,3,1,0
9,2026-05-24 21:01:57,21,6,24,5,1,0


### check feature group

In [8]:
target = "is_fraud"

# identifiers and raw timestamp are kept seprate from model features
identifier_cols = ["transaction_id", "user_id", "timestamp"]

In [ ]:
feature_cols = [col for col in df_fe.columns if col not in identifier_cols + [target]]

In [10]:
X = df_fe[feature_cols].copy()
y = df_fe[target].copy()

In [11]:
print("Feature Count:", len(feature_cols))
print("\nFeature Columns:")
print(feature_cols)

Feature Count: 21

Feature Columns:
['amount', 'merchant_category', 'transaction_type', 'sender_bank', 'receiver_bank', 'device_type', 'os', 'location', 'ip_risk_score', 'device_trust_score', 'failed_attempts', 'transaction_velocity', 'account_age_days', 'is_new_device', 'is_new_location', 'beneficiary_age_days', 'hour', 'is_weekend', 'day_of_week', 'day_of_month', 'month']


In [12]:
print("\nTarget distribution:")
display(y.value_counts())
print("Fraud rate:", round(y.mean() * 100, 3), "%")


Target distribution:


is_fraud
0    98412
1     1588
Name: count, dtype: int64

Fraud rate: 1.588 %


In [13]:
OUTPUT_PATH = PROJECT_DIR / "data" / "upi_fraud_detection_feature_engineered.csv"

In [14]:
df_fe.to_csv(OUTPUT_PATH, index=False)

### Feature Engineering output save

In [15]:
print("Feature engineering completed!")
print("Final shape:", df_fe.shape)
print("Saved to:", OUTPUT_PATH)

Feature engineering completed!
Final shape: (100000, 25)
Saved to: D:\UPI_Fraud_Detection_Project_ML\data\upi_fraud_detection_feature_engineered.csv


### Model Traning

In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    roc_auc_score,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
PROJECT_DIR = Path(r"D:\UPI_Fraud_Detection_Project_ML")
DATA_PATH = PROJECT_DIR / "data" / "upi_fraud_detection_feature_engineered.csv"

In [19]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Fraud rate:", round(df["is_fraud"].mean() * 100, 3), "%")
df.head()

Dataset shape: (100000, 25)
Fraud rate: 1.588 %


,transaction_id,user_id,timestamp,amount,merchant_category,transaction_type,sender_bank,receiver_bank,device_type,os,...,account_age_days,is_new_device,is_new_location,beneficiary_age_days,hour,is_weekend,is_fraud,day_of_week,day_of_month,month
0,TXN00000001,USR001339,2026-07-20 17:27:54,154.25,Food,Bill Payment,Kotak,SBI,Android,Android 15,...,2472,0,0,932,17,0,0,0,20,7
1,TXN00000002,USR011610,2026-08-29 20:08:54,102.42,Utilities,Bill Payment,Axis,Kotak,Android,Android 14,...,163,0,0,1166,20,1,0,5,29,8
2,TXN00000003,USR009819,2026-06-06 02:13:15,324.43,Travel,Merchant Payment,IDFC First,Kotak,Android,Android 15,...,368,0,0,392,2,1,0,5,6,6
3,TXN00000004,USR006584,2026-04-24 00:45:32,42.56,Shopping,Merchant Payment,IndusInd,IDFC First,Android,Android 14,...,953,0,1,490,0,0,0,4,24,4
4,TXN00000005,USR006496,2026-01-02 06:16:25,42.32,Education,Bank Transfer,IndusInd,ICICI,Android,Android 15,...,104,0,0,665,6,0,0,4,2,1


### Fraud And Target 

In [ ]:
target = "is_fraud"
drop_cols = ["transaction_id", "user_id", "timestamp", target]

X = df.drop(columns=drop_cols)
y = df[target]

In [ ]:
categorical_cols = X.select_dtypes(
    include=["object", "str", "category"]
).columns.tolist()

In [ ]:
numerical_cols = X.select_dtypes(include=["number"]).columns.tolist()

In [ ]:
print("Total features:", X.shape[1])
print("Categorical features:", categorical_cols)
print("Numeric feature count:", len(numerical_cols))
print("Target Distribution:")
y.value_counts()

Total features: 21
Categorical features: ['merchant_category', 'transaction_type', 'sender_bank', 'receiver_bank', 'device_type', 'os', 'location']
Numeric feature count: 14
Target Distribution:


is_fraud
0    98412
1     1588
Name: count, dtype: int64

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [26]:
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Training fraud rate:", round(y_train.mean() * 100, 3), "%")
print("Testing fraud rate:", round(y_test.mean() * 100, 3), "%")

Training rows: 80000
Testing rows: 20000
Training fraud rate: 1.588 %
Testing fraud rate: 1.59 %
